Notebook for testing ground truth data processing, plotting, and error calculations

In [ ]:
import os
import csv
import pickle
import logging
import numpy as np
import pandas as pd
import sumolib
from datetime import datetime 
import xml.etree.ElementTree as ET
from scipy.stats import wasserstein_distance
from typing import List, Union
from matplotlib import pyplot as plt
from collections import defaultdict
from scipy.interpolate import LinearNDInterpolator
import time
import ijson
import seaborn as sns

from utils import get_array, cache_trajectories_for_timestamps, get_time_of_day_in_minutes, convert_to_cst_unix, convert_sumo_xy_to_mm

In [ ]:
# load the VelocityGridErrorCalculator class
from find_error_new import VelocityGridErrorCalculator as Error

In [ ]:
# Historical instantiation example for an older error-calculator helper

'''
target_times = ["2022-11-30 06:20:00", "2022-11-20 06:30:00"]
fcd_file_path = "sim_files/0/fcd.xml"
net_file = "sim_files/0/sumo_test.net.xml"
time_interval = 10 # units are seconds
distance_interval = 0.125 # units are miles
EC = Error(target_times, granularity=1)
EC.calculate_avg_sim_speed(fcd_file_path, net_file, time_interval, distance_interval)
'''

In [ ]:
# Historical data-loading example
i24_data_filename = "1130.json"
min_t = pd.to_datetime("2022-11-30 14:00:00.400000095")

i24_data_file_path = os.path.join("../i24_data", i24_data_filename)
westbound_trajectories_1_hr_df = Error.load_trajectories(i24_data_file_path, pd.Timedelta(hours=1), min_t)

In [ ]:
# Historical data-loading example
# Save to csv
westbound_trajectories_1_hr_df.to_csv("data/westbound_trajectories_1_hr_df.csv")
# westbound_trajectories_1_hr_df.to_csv("data/westbound_trajectories_1_hr_df_eastbound.csv")

# load CSV
westbound_trajectories_1_hr_df = pd.read_csv("data/westbound_trajectories_1_hr_df.csv")
# westbound_trajectories_1_hr_df = pd.read_csv("data/westbound_trajectories_1_hr_df_eastbound.csv")

westbound_trajectories_1_hr_df["timestamp"] = pd.to_datetime(westbound_trajectories_1_hr_df["timestamp"])

In [ ]:
print(westbound_trajectories_1_hr_df.describe())

In [ ]:
%store westbound_trajectories_1_hr_df

In [ ]:
%store -r

In [ ]:
# Historical data-loading example
def westbound_lane_func(space_bin):
    if space_bin >= 0 and space_bin <= 5:
        return 5
    if space_bin >= 6 and space_bin <= 13:
        return 4
    else:
        return 5

In [ ]:
# Historical data-loading example
flow_10sec_200m_1hr, density_10sec_200m = Error.get_flow_density_matrix(westbound_trajectories_1_hr_df, westbound_lane_func, time_interval = pd.Timedelta(seconds=10), space_interval = 200, output_filename="10sec_200m_flow_density_1hr.csv")
t_min, t_max = westbound_trajectories_1_hr_df["timestamp"].min(), westbound_trajectories_1_hr_df["timestamp"].max()
x_min, x_max = westbound_trajectories_1_hr_df["x_position"].min(), westbound_trajectories_1_hr_df["x_position"].max()
# x_max -=400
flow_10sec_200m_1hr = flow_10sec_200m_1hr[:,:-1]
flow_10sec_200m_1hr = np.flip(flow_10sec_200m_1hr, axis=1)
#flow_10sec_100m_1hr[:, 2] = (flow_10sec_100m_1hr[:, 1] + flow_10sec_100m_1hr[:, 3])/2

density_10sec_200m = density_10sec_200m[:,:-1]
density_10sec_200m = np.flip(density_10sec_200m, axis=1)
#density_10sec_100m[:, 2] = (density_10sec_100m[:, 1] + density_10sec_100m[:, 3])/2
velocity_arr = flow_10sec_200m_1hr/density_10sec_200m
np.save("flow_10sec_200m_1hr.npy", flow_10sec_200m_1hr)
np.save("density_10sec_200m_1hr.npy", density_10sec_200m)
np.save("velocity_10sec_200m_1hr.npy", velocity_arr)

#Error.plot_matrices(np.flip(flow_10sec_100m_1hr, axis=1), np.flip(density_10sec_100m, axis=1), pd.Timedelta(seconds=10), 400,  t_min, t_max, x_min, x_max)

In [ ]:
#TODO: insert code here to define rho_hat, q_hat, and v_hat. and predictions for each

q_hat = np.load("flow_10sec_200m_1hr.npy")
print(np.max(q_hat))
print(q_hat.shape)
rho_hat = np.load("density_10sec_200m_1hr.npy")
print(rho_hat.shape)
v_hat = q_hat/rho_hat
print(v_hat.shape)

In [ ]:
print("rho mape", Error.mape(rho_hat[:,1:-1], rho_pred_array))
print("q mape", Error.mape(q_hat[:, 1:-1], q_array))
print("v mape", Error.mape(v_hat[:, 1:-1], v_pred_array))
print("----------------")
print("rho rmse", Error.rmse(rho_hat[:,1:-1], rho_pred_array))
print("q rmse", Error.rmse(q_hat[:, 1:-1], q_array))
print("v rmse", Error.rmse(v_hat[:, 1:-1], v_pred_array))

In [ ]:
v_plot = v_pred_array
vmax = print(np.max(v_hat))
plt.figure(figsize=(10, 6))
plt.imshow(v_plot.T, aspect='auto', origin='lower', cmap='RdYlGn', interpolation="none", vmin = 0, vmax = vmax)

# Label axes
plt.xlabel('Time Step')
plt.ylabel('Segment Index')
plt.title('Time-Space Diagram of Velocity (km/h)')
plt.colorbar(label='Flow')

plt.tight_layout()
# plt.show()

In [ ]:
v_plot = v_hat
vmax = print(np.max(v_hat))
plt.figure(figsize=(10, 6))
plt.imshow(v_plot.T, aspect='auto', origin='lower', cmap='RdYlGn', interpolation="none", vmin = 0, vmax = vmax)

# Label axes
plt.xlabel('Time Step')
plt.ylabel('Segment Index')
plt.title('Time-Space Diagram of Velocity Ground Truth')
plt.colorbar(label='Velocity (km/hr)')

plt.tight_layout()
# plt.show()